# 04 Graphene Reference Alignment

This notebook is an optional Graphene reference check. It uses bundled dftbephy Graphene reference data as an external physics/numerical reference, but does not require the dftbephy Python package, HSD inputs, or CLI workflow.


In [ ]:
from pathlib import Path
import json

import numpy as np
from scipy import linalg

from dptb.postprocess.unified.eph.contraction import compute_coupling_matrix
from dptb.postprocess.unified.eph import providers as eph_providers
from dptb.postprocess.unified.eph.providers import _primitive_to_supercell_from_supercell_to_primitive
from dptb.postprocess.unified.eph.utils import reshape_phonopy_eigenvectors

ROOT = Path.cwd()
DATA = ROOT / "data"
GRAPHENE = DATA / "graphene"
REFERENCE = GRAPHENE / "reference"
WORK = ROOT / "work"
WORK.mkdir(exist_ok=True)

required = [
    GRAPHENE / "phonons" / "phonopy_disp.yaml",
    GRAPHENE / "phonons" / "FORCE_SETS",
    REFERENCE / "reference.npz",
    REFERENCE / "derivatives.npz",
    REFERENCE / "alignment_reference.npz",
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. The bundled Graphene reference fixture is incomplete.")

import phonopy

STD_ORBITAL_ORDER = {"s": ["s"], "p": ["py", "pz", "px"], "d": ["dxy", "dyz", "dz2", "dxz", "dx2-y2"]}
THZ_TO_EV_DFTBEPHY = 0.00413566553853599
HBAR_AMU_THZ_DFTBEPHY = 6.35


## Load reference data

The reference arrays are used only to check the DeePTB contraction convention: electronic eigenvalues at `k`, eigenvalues at `k+q`, and `g2` / `coupling_strength`.


In [ ]:
phonons = phonopy.load(
    str(GRAPHENE / "phonons" / "phonopy_disp.yaml"),
    force_sets_filename=str(GRAPHENE / "phonons" / "FORCE_SETS"),
)
primitive = phonons.primitive
supercell = phonons.supercell
num_supercells = int(round(abs(np.linalg.det(supercell.supercell_matrix))))
num_primitive_atoms = len(primitive)

supercell_atom_to_cell = np.tile(np.arange(num_supercells), num_primitive_atoms)
primitive_map = supercell.u2u_map
supercell_to_primitive_atom = np.array([primitive_map[atom] for atom in supercell.s2u_map], dtype=int)
primitive_to_supercell_atom = _primitive_to_supercell_from_supercell_to_primitive(
    supercell_to_primitive_atom,
    num_primitive_atoms,
    supercell_atom_to_cell=supercell_atom_to_cell,
    preferred_cell=(num_supercells - 1) // 2,
)

angular_momenta = {"C": ["s", "p"]}
supercell_orbitals = [
    sum([STD_ORBITAL_ORDER[angular] for angular in angular_momenta[symbol]], [])
    for symbol in supercell.symbols
]
supercell_orbital_offsets = np.insert(np.cumsum([len(orbs) for orbs in supercell_orbitals]), 0, 0)
primitive_orbitals = [supercell_orbitals[index] for index in primitive_to_supercell_atom]
primitive_orbital_offsets = np.insert(np.cumsum([len(orbs) for orbs in primitive_orbitals]), 0, 0)
orbital_slices = [
    (int(primitive_orbital_offsets[index]), int(primitive_orbital_offsets[index + 1]))
    for index in range(num_primitive_atoms)
]
shortest_vectors, vector_multiplicity = eph_providers._phonopy_supercell_maps(phonons)[3:]

with np.load(REFERENCE / "reference.npz", allow_pickle=False) as data:
    reference_hamiltonian = data["H0"]
    reference_overlap = data["S0"]
with np.load(REFERENCE / "derivatives.npz", allow_pickle=False) as data:
    h_derivatives_supercell = data["H_derivs"]
    overlap_derivatives_supercell = data["S_derivs"]
with np.load(REFERENCE / "alignment_reference.npz", allow_pickle=False) as data:
    qpoints = data["qpoints"]
    frequencies = data["frequencies_ev"] / THZ_TO_EV_DFTBEPHY
    reference_strength = data["reference_strength"]
    kpoint = data["kpoint"]
    reference_eigenvalues_k = data["reference_eigenvalues_k"]
    reference_eigenvalues_kq = data["reference_eigenvalues_kq"]

phonons.run_qpoints(qpoints, with_eigenvectors=True)
phonon_eigenvectors = reshape_phonopy_eigenvectors(
    phonons.get_qpoints_dict()["eigenvectors"],
    num_primitive_atoms,
)
print("qpoints", qpoints.shape)
print("reference g2", reference_strength.shape)


## Recompute DeePTB contraction inputs

This cell reconstructs the benchmark Fourier matrices with the same convention used by the bundled reference data. The EPC contraction and phase/mass/frequency convention under check is DeePTB's `compute_coupling_matrix(...)`.


In [ ]:
def lattice_ft(matrix, kvec):
    norbitals = primitive_orbital_offsets[len(primitive_to_supercell_atom)]
    hk = np.zeros((norbitals, norbitals), dtype=complex)
    for source_atom in primitive_to_supercell_atom:
        primitive_source = supercell_to_primitive_atom[source_atom]
        source_cell = supercell_atom_to_cell[source_atom]
        for target_atom in range(len(supercell_to_primitive_atom)):
            primitive_target = supercell_to_primitive_atom[target_atom]
            target_cell = supercell_atom_to_cell[target_atom]
            multiplicity = vector_multiplicity[target_cell, supercell_to_primitive_atom[source_cell]]
            vectors = shortest_vectors[target_cell, supercell_to_primitive_atom[source_cell], :multiplicity]
            phase = np.exp(2j * np.pi * np.dot(vectors, kvec)).sum() / multiplicity
            src_slice = slice(supercell_orbital_offsets[source_atom], supercell_orbital_offsets[source_atom + 1])
            tgt_slice = slice(supercell_orbital_offsets[target_atom], supercell_orbital_offsets[target_atom + 1])
            out_src = slice(primitive_orbital_offsets[primitive_source], primitive_orbital_offsets[primitive_source + 1])
            out_tgt = slice(primitive_orbital_offsets[primitive_target], primitive_orbital_offsets[primitive_target + 1])
            hk[out_src, out_tgt] += matrix[src_slice, tgt_slice] * phase
    return hk


def lattice_ft_derivative(derivatives, kvec):
    norbitals = primitive_orbital_offsets[len(primitive_to_supercell_atom)]
    dhdr = np.zeros((3, norbitals, norbitals), dtype=complex)
    for source_atom in primitive_to_supercell_atom:
        primitive_source = supercell_to_primitive_atom[source_atom]
        source_cell = supercell_atom_to_cell[source_atom]
        for target_atom in range(len(supercell_to_primitive_atom)):
            primitive_target = supercell_to_primitive_atom[target_atom]
            target_cell = supercell_atom_to_cell[target_atom]
            multiplicity = vector_multiplicity[target_cell, supercell_to_primitive_atom[source_cell]]
            vectors = shortest_vectors[target_cell, supercell_to_primitive_atom[source_cell], :multiplicity]
            phase = np.exp(2j * np.pi * np.dot(vectors, kvec)).sum() / multiplicity
            src_slice = slice(supercell_orbital_offsets[source_atom], supercell_orbital_offsets[source_atom + 1])
            tgt_slice = slice(supercell_orbital_offsets[target_atom], supercell_orbital_offsets[target_atom + 1])
            out_src = slice(primitive_orbital_offsets[primitive_source], primitive_orbital_offsets[primitive_source + 1])
            out_tgt = slice(primitive_orbital_offsets[primitive_target], primitive_orbital_offsets[primitive_target + 1])
            dhdr[:, out_src, out_tgt] += derivatives[primitive_source, :, src_slice, tgt_slice] * phase
    return dhdr

hamiltonian_k = lattice_ft(reference_hamiltonian, kpoint)
overlap_k = lattice_ft(reference_overlap, kpoint)
eigenvalues_k, eigenvectors_k = linalg.eigh(hamiltonian_k, b=overlap_k)
h_derivatives_k = lattice_ft_derivative(h_derivatives_supercell, kpoint)[None]
overlap_derivatives_k = lattice_ft_derivative(overlap_derivatives_supercell, kpoint)[None]

eigenvalues_kq = []
eigenvectors_kq = []
h_derivatives_kq = []
overlap_derivatives_kq = []
for qpoint in qpoints:
    kqpoint = kpoint + qpoint
    hamiltonian_kq = lattice_ft(reference_hamiltonian, kqpoint)
    overlap_kq = lattice_ft(reference_overlap, kqpoint)
    bands_kq, states_kq = linalg.eigh(hamiltonian_kq, b=overlap_kq)
    eigenvalues_kq.append(bands_kq)
    eigenvectors_kq.append(states_kq)
    h_derivatives_kq.append(lattice_ft_derivative(h_derivatives_supercell, kqpoint))
    overlap_derivatives_kq.append(lattice_ft_derivative(overlap_derivatives_supercell, kqpoint))


## Compare against dftbephy / paper Graphene reference

The tolerances match the opt-in pytest benchmark. A passing cell means the DeePTB contraction reproduces the reference eigenvalues and `g2` values for this Graphene fixture.


In [ ]:
_, coupling_strength = compute_coupling_matrix(
    eigenvalues_k=eigenvalues_k[None],
    eigenvectors_k=eigenvectors_k[None],
    eigenvalues_kq=np.asarray(eigenvalues_kq)[:, None, :],
    eigenvectors_kq=np.asarray(eigenvectors_kq)[:, None, :, :],
    h_derivatives_k=h_derivatives_k,
    h_derivatives_kq=np.asarray(h_derivatives_kq)[:, None, :, :, :],
    overlap_derivatives_k=overlap_derivatives_k,
    overlap_derivatives_kq=np.asarray(overlap_derivatives_kq)[:, None, :, :, :],
    phonon_eigenvectors=phonon_eigenvectors,
    masses=np.asarray(supercell.masses)[primitive_to_supercell_atom],
    frequencies=frequencies,
    qpoints=qpoints,
    scaled_positions=primitive.scaled_positions,
    orbital_slices=orbital_slices,
    derivative_mode="row",
    prefactor_amu_thz=HBAR_AMU_THZ_DFTBEPHY,
)

metrics = {
    "max_abs_eigenvalues_k_error_ev": float(np.max(np.abs(eigenvalues_k - reference_eigenvalues_k))),
    "max_abs_eigenvalues_kq_error_ev": float(np.max(np.abs(np.asarray(eigenvalues_kq) - reference_eigenvalues_kq))),
    "max_abs_g2_error_ev2": float(np.max(np.abs(coupling_strength[:, 0] - reference_strength))),
    "reference_shape": list(reference_strength.shape),
    "deeptb_shape": list(coupling_strength[:, 0].shape),
}

np.testing.assert_allclose(eigenvalues_k, reference_eigenvalues_k, atol=1e-12, rtol=0.0)
np.testing.assert_allclose(np.asarray(eigenvalues_kq), reference_eigenvalues_kq, atol=1e-12, rtol=0.0)
np.testing.assert_allclose(coupling_strength[:, 0], reference_strength, atol=5e-12, rtol=1e-12)

(WORK / "graphene_reference_alignment.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))


## Figure: reference alignment

A direct visual check that DeePTB reproduces the bundled Graphene reference `g2` values.

In [ ]:
import matplotlib.pyplot as plt

ref = reference_strength.reshape(-1)
deeptb = coupling_strength[:, 0].reshape(-1)
error = deeptb - ref
fig, (ax_scatter, ax_err) = plt.subplots(1, 2, figsize=(8.2, 3.2), dpi=140)
ax_scatter.scatter(ref, deeptb, s=8, alpha=0.5)
lims = [float(min(ref.min(), deeptb.min())), float(max(ref.max(), deeptb.max()))]
ax_scatter.plot(lims, lims, color="0.2", lw=1.0)
ax_scatter.set_xlabel("reference g2 [eV^2]")
ax_scatter.set_ylabel("DeePTB g2 [eV^2]")
ax_scatter.set_title("g2 parity")
ax_scatter.grid(alpha=0.25)
ax_err.hist(error, bins=40, color="tab:orange", alpha=0.85)
ax_err.set_xlabel("DeePTB - reference [eV^2]")
ax_err.set_ylabel("count")
ax_err.set_title("alignment error")
ax_err.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(WORK / "figure_04_reference_alignment.png", dpi=200)
plt.show()